vLLM 推理引擎的执行逻辑可以概括为一个精密的“三进程协作系统”，其核心目标是在高并发场景下，最大化 GPU 的利用率和吞吐量。它通过解耦<b>请求处理</b>、<b>任务调度</b>和<b>计算执行</b>，实现了高效的连续批处理和显存管理

## 🏗️ 顶层架构：三进程协作

vLLM 的高性能源于其清晰的三层架构，每个层级各司其职，通过高效的进程间通信（IPC）协同工作。

1. <font color='red'>API 服务器进程 (API Server Process) - 请求处理</font>

 这是面向用户的入口，负责处理网络 I/O 和协议转换。
 - 职责：提供一个与 OpenAI API 完全兼容的接口，接收用户的推理请求。它使用异步编程模型（AsyncIO）来高效处理大量并发连接，并负责将文本通过 Tokenizer 编码为 Token ID，以及将生成的 Token ID 解码回文本，实现流式输出。
 - 核心价值：将繁重的 I/O 操作与 GPU 计算分离，确保即使 GPU 正在满负荷计算，前端也能快速响应新请求，维持服务的流畅性。
 
2. <font color='red'>引擎进程 (Engine Process) - 任务调度</font>

 这是 vLLM 的“大脑”和“指挥中心”，包含了最核心的调度和内存管理逻辑。
 - 职责：作为推理任务的控制中枢，<font color='red'>它接收来自 API 服务器的请求，并决定在何时、以何种方式执行它们。</font>其内部的 调度器（Scheduler） 和 KV 缓存管理器（KVCacheManager） 是实现高性能的关键。
 - 核心价值<font color='red'>：实现了连续批处理和分页注意力机制，从根本上解决了 GPU 算力浪费和显存碎片化的问题。</font>
 
3. <font color='red'>GPU 工作进程 (GPU Worker Process) - 计算执行</font>

 这是最终的“执行者”，负责在 GPU 硬件上执行实际的计算。
 - 职责：接收来自引擎进程的指令和数据（如 Token ID 和内存块表），加载模型权重，并调用高度优化的 CUDA 内核（如 PagedAttention 内核）<font color='red'>执行模型的前向传播，生成下一个 Token 的概率分布。</font>
 - 核心价值：专注于纯粹的数学计算，通过算子融合等技术，将计算效率推向极致。

## ⚙️ 核心工作流程：一个请求的生命周期

让我们追踪一个用户请求“请介绍法国的首都”在 vLLM 内部的完整旅程：
1. <font color='red'>请求接收与解析</font>

请求通过 OpenAI 兼容接口进入 API 服务器进程。AsyncLLMEngine 接收请求，调用 Tokenizer 将其编码为 Token ID 序列，然后通过 IPC 将请求发送给 引擎进程。

2. <font color='red'>调度与内存分配</font>

引擎进程 的核心 EngineCore 将请求放入 调度器（Scheduler） 的等待队列中。调度器采用 <font color='red'>连续批处理（Continuous Batching） 策略</font>，<b>它不会等待当前批次的所有请求都完成，而是在每个执行步骤（step）中，动态地将等待队列中的新请求和正在生成的旧请求组合成一个新的批次，以填满 GPU 的计算能力。</b>
当一个请求被调度选中时，KV 缓存管理器（KVCacheManager） 会为其分配存储 KV 缓存所需的物理显存块。它利用 PagedAttention 思想，将显存划分为固定大小的块，按需分配，避免了传统方法中因预分配连续显存而导致的碎片化和浪费。

3. <font color='red'>模型执行</font>

引擎进程将组织好的批次数据（包括 Token ID 和描述显存块分布的“块表”）发送给 GPU 工作进程。ModelRunner 根据块表组装输入张量，并调用 PagedAttention CUDA 内核<font color='red'> 在 GPU 上执行模型的前向传播</font>。<b>这个内核能够高效地处理物理上不连续但逻辑上连续的 KV 缓存，完成注意力计算。</b>

4. <font color='red'>采样与回传</font>

<font color='red'>GPU 计算出下一个 Token 的概率分布（logits）后，将其传回 引擎进程。引擎中的采样器根据温度（Temperature）、Top-P 等参数采样出具体的 Token ID</font>。这个新生成的 Token 会被更新到序列中，如果需要更多显存，会再次向 KV 缓存管理器申请。最终，生成的 Token ID 通过 IPC 返回给 API 服务器进程，被解码为文本并流式地发送给用户。

## 💡 两大核心技术

vLLM 的高性能源于两个革命性的设计：

### 连续批处理 (Continuous Batching)

传统的静态批处理必须等待批次中所有请求都完成才能处理新请求，导致 GPU 在长短不一的请求面前效率低下。vLLM 的调度器实现了迭代级别的批处理，允许正在解码（Decode）的老请求和新进入预填充（Prefill）的请求在同一个批次中混合执行。当一个请求生成结束，它占用的计算资源会立即被新请求填补，从而始终保持 GPU 的高利用率，显著提升了吞吐量并降低了延迟。

### 分页注意力 (PagedAttention)

这是 vLLM 的“杀手锏”。它将操作系统中虚拟内存分页的思想引入到注意力计算中。
- <font color='red'>问题：</font>传统方法为每个序列预分配一块连续的显存来存储 KV 缓存，这不仅导致严重的显存碎片，还容易因预估不准而引发 OOM（显存溢出）。
- <font color='red'>解决方案：</font>PagedAttention 将 KV 缓存切分成固定大小的“块”（Block）。KV 缓存管理器维护一个全局的空闲块池，并为每个序列维护一个“块表”，记录其逻辑块到物理显存块的映射。这样，一个序列的 KV 缓存可以分散在物理上不连续的显存块中。PagedAttention 的 CUDA 内核在执行计算时，会根据块表高效地 gather 所需的数据。
- <font color='red'>优势：</font>这种方法几乎完全消除了显存碎片，极大地提高了显存利用率，使得 vLLM 能够处理更长的上下文和更高的并发请求。